<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

## 05.06 Understanding Training

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 1. Introduction

So far we have always started from a *finished* model: [`05.05_Introduction_to_LLMs.ipynb`](./05.05_Introduction_to_LLMs.ipynb) showed how such a model turns probabilities into text and the notebooks around the Raspberry Pi showed how to run one on a small device. This short notebook fills the gap in between: **where do the weights come from?**

In the middle of this notebook, a genuinely tiny language model goes from random noise to readable sentences, live, in about half a minute on your laptop. Every ingredient it uses is one you have already met: token embeddings ([`05.02_Understanding_Word_Embedding.ipynb`](./05.02_Understanding_Word_Embedding.ipynb)), attention ([`05.03_Understanding_Attention.ipynb`](./05.03_Understanding_Attention.ipynb)), and next-token probabilities ([`05.05_Introduction_to_LLMs.ipynb`](./05.05_Introduction_to_LLMs.ipynb)). Training is the process that fills in their values, and conceptually it is the same process at every scale: our model and a frontier model differ in size by a factor of about a million, not in kind.

<div class="alert alert-block alert-success">

Make sure that you have installed all necessary packages. If you are missing some, you can either use `pip install` or the provided cell in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb) (run the cell in section 1.2: "Install the Course Libraries").
</div>

### 2. The Objective: Measuring Surprise

Training needs a single number to improve, and for language models that number is quite simple. The model reads text and, at every position, assigns a probability to each possible next character. The **loss** measures how *surprised* the model is when the true next character arrives:

$$
\text{loss} \;=\; -\log_2\, p(\text{true next character})
$$

Measured this way, the unit of surprise is the **bit**, the same currency you met in [`01.5_Entropy.ipynb`](../01_Data_and_Signals/01.5_Entropy.ipynb). A model that assigns the true character a probability of 0.5 pays exactly one bit, the surprise of one fair coin flip. Averaged over a whole text, the loss says how many bits of surprise the model pays *per character* and training is nothing more than the attempt to push this average down.

Our model later in this notebook chooses among 28 characters. Three imaginary models show how the loss behaves before we train anything:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

VOCAB_SIZE = 28   # the character set our tiny model will use later

scenarios = {
    "clueless: spreads probability evenly": 1 / VOCAB_SIZE,
    "confident and right: p = 0.9":         0.9,
    "confident and wrong: p = 0.001":       0.001,
}

surprise_bits = {name: -np.log2(p) for name, p in scenarios.items()}

fig, ax = plt.subplots(figsize=(10, 3.2))
bars = ax.barh(list(surprise_bits), list(surprise_bits.values()), height=0.55, color="#4c78a8")
for bar, bits in zip(bars, surprise_bits.values()):
    ax.text(bits + 0.15, bar.get_y() + bar.get_height() / 2, f"{bits:.2f} bits", va="center")

ax.invert_yaxis()
ax.set_xlabel("surprise when the true next character arrives (bits)")
ax.set_xlim(0, 11.5)
ax.set_title("The loss rewards confidence only when it is justified", loc="left")
fig.tight_layout()
plt.show()

### 3. Learning: Millions of Knobs

Two things in the chart are worth holding on to. The clueless model pays $\log_2 28 \approx 4.8$ bits per character. Remember that number: it returns in a few minutes as the starting point of our own training run. And being confidently *wrong* costs far more than admitting ignorance: the loss punishes misplaced certainty, which is exactly the behavior we want to train away.

So how does the loss go down? A model is a pile of adjustable numbers, the weights, and **gradient descent** improves them by repetition: predict a batch of text, measure the surprise, compute for every single weight the direction that would have reduced it, and nudge every weight a small step that way. Then repeat, millions of times. You already watched this loop shape attention patterns in [`05.03_Understanding_Attention.ipynb`](./05.03_Understanding_Attention.ipynb). Here we care about one practical knob of the loop itself.

That knob is the **learning rate**: how large each nudge is. The picture below descends the simplest possible loss landscape with three different settings. One weight, one valley. Real models descend the same way, just in millions of dimensions at once.

In [ ]:
learning_rates = {"too small: lr = 0.02": "#4c78a8",
                  "well chosen: lr = 0.25": "#1b9e77",
                  "too large: lr = 1.05": "#d95f02"}

w_axis = np.linspace(-4.5, 4.5, 200)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)

for ax, (label, color) in zip(axes, learning_rates.items()):
    lr = float(label.split("= ")[1])
    w, path = 3.0, [3.0]
    for _ in range(12):
        w = w - lr * 2 * w          # gradient of loss(w) = w^2 is 2w
        path.append(w)
    path = np.array(path)

    ax.plot(w_axis, w_axis ** 2, color="#bbbbbb", linewidth=1.2, zorder=1)
    ax.plot(path, path ** 2, color=color, linewidth=1.2, marker="o",
            markersize=5, zorder=3)
    ax.plot(path[0], path[0] ** 2, marker="o", markersize=9, color=color, zorder=4)
    ax.set_title(label, loc="left", fontsize=10, color=color)
    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-0.5, 14)
    ax.set_xlabel("the weight being adjusted")

axes[0].set_ylabel("loss")
plt.show()

Each panel starts at the same large dot and takes twelve steps. With a small learning rate the model inches downhill. Every step is compute you pay for, and most of it is wasted. With a well-chosen rate it settles into the valley in a handful of steps. With a rate that is too large, each nudge overshoots the valley and lands *higher* on the opposite slope: the loss explodes and the run is ruined.

At our scale a ruined run costs half a minute. At frontier scale, where a training run occupies thousands of accelerators for months, these knobs guard a small power plant, which is why training is as much an engineering discipline as a data problem and why experienced training teams are scarce and expensive.

### 4. Watching a Language Model Learn

Three ingredients assembled one cell at a time: data, a model, and the loop.

#### 4.1 The data

A model can learn *only* what its training data contains: nothing else, no matter how long we train. To make that visible we build our corpus ourselves, from sentence templates about a world you know well: devices, signals, and wireless links. The vocabulary is deliberately tiny and the sentences deliberately repetitive. This is the same trick behind *TinyStories*, the 110-million-parameter model you ran in [`05.07_Introduction_to_Local_LLMs.ipynb`](./05.07_Introduction_to_Local_LLMs.ipynb): radically simple data lets a radically small model learn coherent language.

In [ ]:
import random

def build_corpus(n_sentences=4000, seed=0):
    rng = random.Random(seed)
    devices = ["laptop", "phone", "router", "sensor", "drone", "tablet", "camera", "robot"]
    links = ["wi-fi", "bluetooth", "lifi", "5g"]
    places = ["office", "kitchen", "garden", "basement", "classroom", "street"]
    templates = [
        "the {d} sends data over {l}.",
        "the {d} in the {p} connects over {l}.",
        "when the signal is weak, the {d} sends the data again.",
        "the {d} measures the signal and reports it.",
        "a strong signal means fast data for the {d}.",
        "the {d} talks to the {d2} over {l}.",
        "in the {p}, the {l} signal is strong.",
        "the {d} waits, then sends a short message over {l}.",
        "the battery of the {d} runs low, so it sends less data.",
        "the {d} listens before it talks on the shared channel.",
    ]
    sentences = []
    for _ in range(n_sentences):
        t = rng.choice(templates)
        sentences.append(t.format(d=rng.choice(devices), d2=rng.choice(devices),
                                  l=rng.choice(links), p=rng.choice(places)))
    return " ".join(sentences)


corpus = build_corpus()

# The characters are our token vocabulary. We add "?" so we can ask questions later.
chars = sorted(set(corpus) | {"?"})
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
VOCAB_SIZE = len(chars)

print(f"corpus: {len(corpus):,} characters")
print(f"vocabulary ({VOCAB_SIZE} characters): {''.join(chars)!r}")
print(f"\nbeginning of the corpus:\n  {corpus[:150]}...")

#### 4.2 The model

The model is a miniature of the real thing, and every part of it points back to an earlier notebook: an embedding table turns each character into a vector, a stack of transformer blocks applies the attention mechanism you built by hand, and a final layer turns the result into a probability for each of the 28 characters. Real LLMs are this, a million times over.

We keep it at about a quarter of a million parameters so that a full training run fits into half a minute of laptop CPU. For comparison, the smallest model on your Pi holds about 400 times more.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

CONTEXT = 48   # how many previous characters the model may look at

class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size, context, d_model=96, n_heads=4, n_layers=2):
        super().__init__()
        self.context = context
        # characters -> vectors (see HandsOn_Understanding_Embeddings)
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(context, d_model)
        # attention blocks (the mechanism from HandsOn_Understanding_Attention)
        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=4 * d_model,
            dropout=0.0, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        # vectors -> a score for every character in the vocabulary
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        T = idx.shape[1]
        h = self.token_embedding(idx) + self.position_embedding(torch.arange(T))
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T)
        h = self.blocks(h, mask=causal_mask, is_causal=True)   # may only look backwards
        return self.head(h)


@torch.no_grad()
def sample_text(model, prompt, n_chars=100, temperature=0.8, seed=1):
    """Generate text character by character, exactly as in HandsOn_Introduction_to_LLMs."""
    generator = torch.Generator().manual_seed(seed)
    model.eval()
    idx = torch.tensor([[stoi[c] for c in prompt]])
    for _ in range(n_chars):
        logits = model(idx[:, -CONTEXT:])
        probs = F.softmax(logits[0, -1] / temperature, dim=-1)
        next_char = torch.multinomial(probs, 1, generator=generator)
        idx = torch.cat([idx, next_char.view(1, 1)], dim=1)
    model.train()
    return "".join(itos[int(i)] for i in idx[0])


model = TinyLanguageModel(VOCAB_SIZE, CONTEXT)
n_parameters = sum(p.numel() for p in model.parameters())
print(f"model parameters : {n_parameters/1e3:.0f} thousand")
print(f"for comparison   : gemma3:270m has {0.27e9/n_parameters:,.0f}x more")

#### 4.3 The training loop

The loop below is the whole secret. Four steps, repeated: **draw** a random batch of text snippets, **predict** every next character, **measure** the surprise in bits, and **nudge** every weight downhill. Labs run this exact loop, with more data, more weights, and a few years of stabilizing tricks, for months on end.

So that we can watch the model learn, the loop pauses at a few checkpoints and lets the model generate text from the prompt `"the "`. Watch what it produces at step 0, and what it produces two hundred steps later.

In [ ]:
data = torch.tensor([stoi[c] for c in corpus], dtype=torch.long)

def get_batch(batch_size=64):
    starts = torch.randint(len(data) - CONTEXT - 1, (batch_size,))
    x = torch.stack([data[i:i + CONTEXT] for i in starts])        # what the model sees
    y = torch.stack([data[i + 1:i + CONTEXT + 1] for i in starts])  # the true next characters
    return x, y


# Before training starts, record what the model believes follows "... over w".
probe_prompt = "the router sends data over w"
with torch.no_grad():
    model.eval()
    probe_logits = model(torch.tensor([[stoi[c] for c in probe_prompt]]))
    probs_before = F.softmax(probe_logits[0, -1], dim=-1).clone()
    model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
STEPS = 1000
SAMPLE_AT = {0, 15, 60, 150}
LOG2 = float(np.log(2))

import time
losses = []
start_time = time.time()

for step in range(STEPS):
    if step in SAMPLE_AT:
        print(f"--- step {step:4d} ---")
        print(f"  {sample_text(model, 'the ')}\n")

    x, y = get_batch()                                                  # 1. draw
    logits = model(x)                                                   # 2. predict
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1)) / LOG2   # 3. measure
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()                                                    # 4. nudge

    losses.append(float(loss.detach()))
    if step % 200 == 199:
        print(f"step {step + 1:4d}: {np.mean(losses[-50:]):.3f} bits/char   "
              f"({time.time() - start_time:.0f} s elapsed)")

print(f"\n--- step {STEPS} (final) ---")
print(f"  {sample_text(model, 'the ')}")
print(f"\ntrained {STEPS} steps in {time.time() - start_time:.0f} s on this machine")

with torch.no_grad():
    model.eval()
    probe_logits = model(torch.tensor([[stoi[c] for c in probe_prompt]]))
    probs_after = F.softmax(probe_logits[0, -1], dim=-1)
    model.train()

Read the checkpoint samples top to bottom: they are the interesting output of the whole notebook. At step 0 the model produces uniform noise: every character equally likely, 4.8 bits of surprise each. Within a few dozen steps it discovers the cheapest regularities first: which letters are common, that a space follows a word, the shape of `the`. Then come whole words, then word *order*, and only near the end does the grammar settle. Nothing chose this curriculum: frequent, local patterns simply pay off more loss reduction per nudge than rare, global ones, so gradient descent buys them first.

Note also what nobody programmed. No cell of this notebook contains a spelling of `bluetooth` or a rule of grammar. The model extracted both from raw text, because doing so lowered its surprise. That is the entire mechanism by which large models acquire what looks like knowledge.

#### 4.4 The loss curve

The samples showed the journey qualitatively. The loss curve is its instrument reading. This curve (surprise per character, step by step) is the single most watched chart in every AI lab, the pulse monitor of investments.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))

smoothed = np.convolve(losses, np.ones(25) / 25, mode="valid")
ax.plot(losses, color="#1b9e77", alpha=0.25, linewidth=0.8)
ax.plot(np.arange(len(smoothed)) + 12, smoothed, color="#1b9e77", linewidth=1.8,
        label="loss (smoothed)")

ax.axhline(np.log2(VOCAB_SIZE), color="#757575", linestyle="--", linewidth=1.0)
ax.text(len(losses) * 0.55, np.log2(VOCAB_SIZE) + 0.12,
        f"clueless model: log2({VOCAB_SIZE}) = {np.log2(VOCAB_SIZE):.1f} bits", color="#757575")

for rank, step in enumerate(sorted(SAMPLE_AT)):
    ax.axvline(step, color="#bbbbbb", linewidth=0.8, zorder=1)
    ax.text(step + 8, 4.45 - (rank % 2) * 0.85, f"sample\n@ {step}",
            fontsize=8, color="#555555")

ax.set_xlabel("training step")
ax.set_ylabel("loss (bits per character)")
ax.set_ylim(0, 5.2)
ax.set_xlim(0, len(losses))
ax.legend(loc="center right")
ax.set_title("Surprise falls as the data's patterns move into the weights", loc="left")
plt.show()

The curve starts exactly at the clueless line from Section 2: before training, the network *is* the clueless model. The steep early fall is the cheap structure being absorbed and the long flat tail is the expensive remainder. Most of the visible magic happened in the first fifth of the run, and every further step bought less. Keep that shape in mind for Section 5.

#### 4.5 What changed inside

The weights moved, but what does that *mean*? The probability view makes it concrete. In [`05.05_Introduction_to_LLMs.ipynb`](./05.05_Introduction_to_LLMs.ipynb) you inspected next-token probability tables of a trained model; here is the same table for our model, **before and after** training, at the prompt `"the router sends data over w"`. In our corpus, the only link whose name starts with `w` is `wi-fi`, so a model that has truly absorbed the data should be nearly certain the next character is `i`.

In [ ]:
top_after = torch.argsort(probs_after, descending=True)[:6]
labels = ["space" if itos[int(i)] == " " else repr(itos[int(i)]) for i in top_after]
before_values = [float(probs_before[i]) for i in top_after]
after_values = [float(probs_after[i]) for i in top_after]

positions = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(positions - 0.18, before_values, width=0.36, color="#757575", label="before training")
ax.bar(positions + 0.18, after_values, width=0.36, color="#1b9e77", label="after training")

for x, v in zip(positions + 0.18, after_values):
    ax.text(x, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)

ax.set_xticks(positions, labels)
ax.set_ylabel("probability of the next character")
ax.set_ylim(0, 1.12)
ax.legend()
ax.set_title(f'P(next character) after the prompt "{probe_prompt}"', loc="left")
plt.show()

Before training, the probability is spread thinly across the whole alphabet. After training, virtually all of it sits on `i`. Multiply this by every context the model has ever seen and you have the honest description of what training produced: not stored sentences, but millions of sharpened conditional probabilities, compressed into the weights.

### 5. A Tiny Scaling Law

One question dominates every training budget: **what does a bigger model buy?** We can answer it the evidence-based way, on our own problem. The cell below trains four models of increasing size with the *same* recipe (same data, same 300 steps) and records what each one achieved and what it cost in time. (This takes around a minute. The growing per-model runtime *is* part of the measurement.)

In [ ]:
import pandas as pd

configurations = [
    dict(d_model=24,  n_heads=2, n_layers=1),
    dict(d_model=48,  n_heads=2, n_layers=1),
    dict(d_model=96,  n_heads=4, n_layers=2),
    dict(d_model=160, n_heads=4, n_layers=3),
]
SCALING_STEPS = 300

records = []
for config in configurations:
    torch.manual_seed(0)
    candidate = TinyLanguageModel(VOCAB_SIZE, CONTEXT, **config)
    opt = torch.optim.AdamW(candidate.parameters(), lr=3e-3)
    t0 = time.time()
    recent = []
    for step in range(SCALING_STEPS):
        x, y = get_batch()
        loss = F.cross_entropy(candidate(x).reshape(-1, VOCAB_SIZE), y.reshape(-1)) / LOG2
        opt.zero_grad(); loss.backward(); opt.step()
        recent.append(float(loss.detach()))
    records.append({
        "parameters": sum(p.numel() for p in candidate.parameters()),
        "training seconds": round(time.time() - t0, 1),
        "final loss (bits/char)": round(float(np.mean(recent[-30:])), 3),
        "d_model": config["d_model"],
    })
    print(f"d_model={config['d_model']:4d}: done in {records[-1]['training seconds']} s")

scaling = pd.DataFrame(records).set_index("d_model")
print()
print(scaling)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4), constrained_layout=True)
for ax, column, color in [(axes[0], "final loss (bits/char)", "#1b9e77"),
                          (axes[1], "training seconds", "#4c78a8")]:
    ax.plot(scaling["parameters"], scaling[column], color=color, alpha=0.4)
    ax.scatter(scaling["parameters"], scaling[column], s=80, color=color, zorder=3)
    for d_model, row in scaling.iterrows():
        ax.annotate(f"d={d_model}", (row["parameters"], row[column]),
                    textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8)
    ax.set_xscale("log")
    ax.set_xlabel("parameters (log scale)")
    ax.set_ylabel(column)

axes[0].set_ylim(0, None)
axes[1].set_ylim(0, None)
axes[0].set_title("What a bigger model buys ...", loc="left")
axes[1].set_title("... and what it costs", loc="left")
plt.show()

The two panels reproduce the finding that governs the industry's spending. Quality improves steeply at first and then flattens. On our simple corpus the two largest models are nearly indistinguishable, because there is little structure left to absorb. Cost, meanwhile, keeps climbing without any flattening at all. Every real training budget lives in the widening gap between these panels.

Two things generalize from this toy. First, the famous **scaling laws** [01]: for real models on real data the left curve keeps improving for a very long time, but only smoothly and predictably, never for free, which is what makes multi-year AI investment plannable at all. Second, the **compute-optimal balance** [02]: for a fixed budget, a smaller model trained on more data often beats a larger one trained on less. The reason given in [`05.05_Introduction_to_LLMs.ipynb`](./05.05_Introduction_to_LLMs.ipynb), Section 3, why data quality and volume have become the real battleground. And once the expensive training is finished, the pipeline of [`05.08_Making_Models_Smaller.ipynb`](./05.08_Making_Models_Smaller.ipynb) takes over to make the result deployable.

### 6. A Trained Model Is Not Yet an Assistant

One demonstration remains, and it explains the difference between what we just built and the chat models you use. Our model has now thoroughly learned its little world. So let us ask it a question and, while we are at it, mention something its corpus has never contained.

In [ ]:
print("QUESTION PROMPT")
print("  " + sample_text(model, "what is wi-fi? ", n_chars=90))
print()
print("A WORD THE TRAINING DATA NEVER CONTAINED")
print("  " + sample_text(model, "the satellite ", n_chars=90))

Two failures, both instructive.

The model does not *answer* the question. It **continues** it, drifting into corpus-flavored prose. Nothing in its training ever rewarded answering; the loss only ever rewarded plausible continuation. A freshly pretrained model of any size behaves this way, which is why a raw **base model** is not a product.

And the model handles `satellite` (a word it has never seen) without the slightest hesitation. Fluency is what it was trained for, truth and coverage were never part of the loss. This is the root of what is popularly called **hallucination** and it is why a polished demo tells you little about what a model actually knows.

Turning a continuation machine into an assistant takes further, cheaper training stages on top of pretraining:

| Stage | Data it consumes | Rough scale | What it teaches |
|:------|:-----------------|:------------|:----------------|
| **Pretraining** | trillions of tokens of web text, books, code | months on thousands of accelerators. Public estimates start in the tens of millions of dollars | language, facts, style: to *continue* any text |
| **Instruction tuning (SFT)** | thousands to millions of curated question-answer dialogues | days on a small cluster | to *answer* rather than continue |
| **Preference alignment (RLHF / DPO)** | human rankings of candidate answers | comparable to instruction tuning | to be helpful, harmless, and honest about uncertainty |

What we did in Section 4 was pretraining in miniature: the expensive, foundational stage. The two stages that follow were covered in detail in [`05.12_Understanding_Reasoning_Models.ipynb`](./05.12_Understanding_Reasoning_Models.ipynb), Section 3, and everything after the pipeline you already know: compression for deployment ([`05.08_Making_Models_Smaller.ipynb`](./05.08_Making_Models_Smaller.ipynb)) and decoding control at inference time ([`05.11_Understanding_Inference_Control.ipynb`](./05.11_Understanding_Inference_Control.ipynb)). In the project [`05.16_Build_Your_Own_Model.ipynb`](./05.16_Build_Your_Own_Model.ipynb) you will walk this pipeline yourself, at small scale, end to end.

### 7. Playground

The trained model is still loaded. Probe it. The point of this section is to build an instinct for what lives in the weights and what does not.

In [ ]:
# Try your own prompts (all characters must exist in the vocabulary printed in Section 4.1).
# Ideas: prompts from the corpus world, half-finished sentences, words the corpus
# never contained, temperature 0.1 versus 1.5.

print(sample_text(model, "when the signal is ", n_chars=100, temperature=0.8))

If you want to go further, rerun Section 4.3 with more steps, or give the model a larger `d_model`. You now own the entire training pipeline, and every improvement you observe is one you caused and can explain.

### 8. Summarizing Remarks

Training a language model is conceptually small and industrially enormous. The loss measures surprise in bits per character; gradient descent nudges millions of weights to reduce it; and everything a model "knows" — spelling, grammar, facts, style — is nothing but patterns that migrated from the training data into the weights because absorbing them lowered the surprise. What the data does not contain, the model cannot know, though it will speak about it fluently all the same.

Scale buys quality smoothly and predictably, but at a cost that climbs without flattening, which is why compute-optimal balancing of model size against data volume decides real budgets. And pretraining alone yields a continuation machine, not an assistant: instruction tuning and preference alignment are what turn one into the other.

For a manager, the practical consequence is liberating: you will almost certainly never pretrain a model, because the economics reserve that stage for a handful of labs. Your decisions live downstream: choosing a model, fine-tuning it, or grounding it in your own documents with retrieval. Those options are exactly where this course goes next. What this notebook gives you is the ability to reason about what you are buying: you have watched the machine being made.

### Further Reading and Literature

[01] Kaplan, J., McCandlish, S., Henighan, T., Brown, T. B., et al. (2020). **Scaling Laws for Neural Language Models.** arXiv. [Paper](https://arxiv.org/abs/2001.08361)

[02] Hoffmann, J., Borgeaud, S., Mensch, A., et al. (2022). **Training Compute-Optimal Large Language Models.** NeurIPS. [Paper](https://arxiv.org/abs/2203.15556)

[03] Ouyang, L., Wu, J., Jiang, X., et al. (2022). **Training language models to follow instructions with human feedback (InstructGPT).** NeurIPS. [Paper](https://arxiv.org/abs/2203.02155)

[04] Eldan, R., & Li, Y. (2023). **TinyStories: How Small Can Language Models Be and Still Speak Coherent English?** arXiv. [Paper](https://arxiv.org/abs/2305.07759). The idea behind our template corpus, and behind the `tinyStories:110m` model on your Raspberry Pi.

[05] Karpathy, A. (2023). [**Let's build GPT: from scratch, in code, spelled out.**](https://www.youtube.com/watch?v=kCc8FmEb1nY) YouTube video lecture. Our tiny model follows the spirit of this walkthrough.

[06] Sanderson, G. (3Blue1Brown). [**Gradient descent, how neural networks learn.**](https://www.youtube.com/watch?v=IHZwWFHWa-w) YouTube video lecture.

----
### Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. Many questions ask for concrete values or outputs from your own notebook run; generic answers without run-specific evidence may receive reduced credit. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**Report the first and the final loss value from your training run. The first one matches a number computed long before training started. Which one, and why must that be so?** Answer:

(add your answer here)

**Describe what happened in each of the three learning-rate panels. Why is a learning rate that is too large a more expensive mistake than one that is too small?** Answer:

(add your answer here)

**Paste one checkpoint sample from early, middle, and late in your run. What did the model master first: letters, words, or word order? Give a reason for that order.** Answer:

(add your answer here)

**Nowhere in this notebook is the spelling of `bluetooth` or any rule of grammar written down, yet the trained model produces both. Where did they come from, and what does that imply about where a large model's abilities and biases come from?** Answer:

(add your answer here)

**Report your scaling table (parameters, seconds, final loss). Between which two sizes was the extra quality per extra second smallest? What do the compute-optimal results [02] recommend when buying quality: only a larger model, or something else?** Answer:

(add your answer here)

**What did the model do with the question `what is wi-fi?`, and which stage of the pipeline in Section 6 is responsible for fixing exactly this behavior?** Answer:

(add your answer here)

**The model wrote fluently about `the satellite`, which its corpus never contained. Explain why this happens, and what it means for judging any language model by a demonstration.** Answer:

(add your answer here)

**Your company wants an assistant that knows its internal documents. Rank the three options: pretrain from scratch, fine-tune an open-weights model, or retrieval over documents (RAG). Use the cost table from Section 6 to justify your ranking.** Answer:

(add your answer here)

**Optional: change one thing in the pipeline (more training steps, a different temperature in the playground, a template of your own in the corpus) and report what changed in the output and why.** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)